In [1]:
!pip install geopandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/32.5 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━ 26.7/32.5 MB 137.5 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.5/32.5 MB 87.5 MB/s  0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/2 [pyogrio]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/2 [pyogrio]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/2 [pyogrio]

   ━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━ 1/2 [geopandas]

   ━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━ 1/2 [geopandas]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [geopandas]


In [2]:
import requests
import pandas as pd
import geopandas as gpd
import time

from urllib.parse import urlencode
from datetime import datetime, timezone, date, timedelta
from zoneinfo import ZoneInfo
from arcgis.gis import GIS
from arcgis.features import GeoAccessor, GeoSeriesAccessor

## Authentication

In [3]:
# ArcGIS account authentication
# If not run from within ArcGIS Online, additional steps are needed to sign in
gis = GIS("home")

## Helper functions

In [4]:
# API base link
BASE = "https://api.inaturalist.org/v1"

# API endpoint
END = "observations"

# Helper functions
def get_inat(endpoint, params = None):
    """
    Call an iNaturalist v1 endpoint and return parsed JSON.
    Raises an error for non-200 responses.
    """
    url = f"{BASE}/{endpoint.lstrip('/')}"
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def obs_to_row(obs):
    """
    Helper function for obs_to_df.
    Input is an observation from get_inat output.
    Returns a row with column names to be used in a data frame.
    """
    taxon = obs.get("taxon") or {}
    coords = obs.get("geojson", {}).get("coordinates")
    photos = obs.get("photos", [])
    image_url = None
    if photos:
        image_url = photos[0].get("url")
        if image_url:
            image_url = image_url.replace("square", "original")

    row = {"id": obs.get("id"),
           "observed_on": obs.get("observed_on"),
           "observed_datetime": datetime.strptime(f"{obs.get("observed_on")} 04:00", "%Y-%m-%d %H:%M"),
           "url": obs.get("uri"),
           "image_url": image_url,
           "description": obs.get("description"),
           "species_guess": obs.get("species_guess"),
           "scientific_name": taxon.get("name"),
           "common_name": taxon.get("preferred_common_name"),
           "iconic_taxon_name": taxon.get("iconic_taxon_name"),
           "longitude": coords[0] if coords else None,
           "latitude": coords[1] if coords else None}
    return row

def obs_to_df(data):
    """
    Takes in a parsed JSON from get_inat.
    Returns a readable dataframe with all columns of interest.
    """
    return pd.DataFrame([obs_to_row(obs) for obs in data["results"]])

def clip(df, poly, crs="EPSG:4326"):
    """
    Takes in a dataframe with latitude/longitude columns, a geo 
    data frame (polygon) specifying the AOI and a CRS code.
    Returns a geo data frame containing only points within the
    AOI.
    """
    points = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df.longitude, df.latitude),
        crs=crs)

    points_clipped = gpd.clip(points, poly)
    return pd.DataFrame.spatial.from_geodataframe(points_clipped)


## Main function

In [5]:
# Bounding box parameter specifying the general AOI
BBOX = {
    "swlat": 43.6538191721394,
    "swlng": -79.33150817493927,
    "nelat": 43.68238158155382,
    "nelng": -79.2764049496256
}

# Main get function
def fetch_all(endpoint, date, per_page=200):
    """
    Input arguments are desired endpoint, date of interest,
    and results per page (by default 200).
    WARNING: iNaturalist API can at most handle 200 results
    per API call.
    Returns a data frame containing all observations created on a
    given date.
    """
    # API parameters
    params = {
        **BBOX,
        "per_page": per_page,
        "page": 1,
        "quality_grade": "research",
        "created_on": date
    }

    # First API call
    data = get_inat(endpoint, params=params)
    total_res = data["total_results"]
    print(f"{total_res} new observations created on {date}")
    
    # If less than per_page results, a single API call required
    if total_res <= per_page:
        return obs_to_df(data)

    # If too many results for a single API call
    res = [obs_to_df(data)]
    num_items = len(data["results"])
    params["page"] = 2
    
    while num_items > 0:
        data = get_inat("observations", params=params)
        df = obs_to_df(data)
        res.append(df)

        params["page"] += 1
        num_items = len(data["results"])

        # delay to reduce risk of overloading the API
        time.sleep(1)

    combined_df = pd.concat(res, ignore_index=True)
    return combined_df


## Main script

In [6]:
# Loading in polygon specifying the Beach BIA neighbourhood
nb = gpd.read_file('/arcgis/home/BeachBIA/Neighbourhoods - 4326.shp')
BEACH = nb[nb["_id1"] == 93]

# ID of the iNaturalist layer to update
# May change after ownership transfer
LAYER_ID = "8f543676b0cf48a2b7df73f96529a7ce"
TIME_LAYER_ID = "111e0bb2780c470a9ee950471bc21375"

# Today and yesterday's date (EST)
est_datetime = datetime.now(ZoneInfo("America/New_York"))
today = est_datetime.date()
yesterday = today - timedelta(days=1)

# Main script
# Fetching all observations created yesterday as a data frame
df = fetch_all(END, yesterday)

if len(df) == 0:
    print("Nothing to update")
else:
    # Clip and convert data frame to geo data frame
    gdf = clip(df, BEACH)
    n_obs = len(gdf)
    
    if n_obs > 0:
        # Getting layer
        layer = gis.content.get(LAYER_ID).layers[0]
        time_layer = gis.content.get(TIME_LAYER_ID).layers[0]
    
        # Updating layer with new results
        layer.edit_features(
            adds = gdf.spatial.to_featureset())
        time_layer.edit_features(
            adds = gdf.spatial.to_featureset())
        
        print(f"Added {n_obs} new observations")
        print("Layer successfully updated")
        
    else:
        print("No new observations in The Beach")

print(f"Script completed as of {datetime.now(ZoneInfo("America/New_York")).strftime("%Y-%m-%d %H:%M")} (EST)")

10 new observations created on 2026-04-20


Added 1 new observations
Layer successfully updated
Script completed as of 2026-04-21 01:00 (EST)
